In [28]:
import pandas as pd
import numpy as np


In [29]:
df = pd.read_csv('../data/results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [30]:
df['date'] = pd.to_datetime(df['date'])

In [31]:
df = df.sort_values('date').reset_index(drop=True)

In [32]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 0                            # Home win
    elif row['home_score'] < row['away_score']:
        return 2                            # away win
    else:
        return 1                               # Draw

In [33]:
df['result'] = df.apply(get_result, axis=1)

In [34]:
df['result'].value_counts()

result
0    24163
2    13935
1    11280
Name: count, dtype: int64

In [35]:
team_history = {}

home_form_points = []

away_form_points = []

home_goals_scored = []

away_goals_scored = []

In [36]:
for _,row in df.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    
    if home_team not in team_history:
        team_history[home_team] = []
    if away_team not in team_history:
        team_history[away_team] = []
    
    home_history = team_history[home_team]
    away_history = team_history[away_team]
    
    home_last5 = home_history[-5:]
    away_last5 = away_history[-5:]
    
    home_points = sum(
        match['points']
        for match in home_last5
    )
    
    away_points = sum(
        match["points"]
        for match in away_last5
    )
    
    home_gf = sum(
        match["goals_scored"]
        for match in home_last5
    )
    
    away_gf = sum(
        match["goals_scored"]
        for match in away_last5
    )
    
    home_form_points.append(home_points)
    away_form_points.append(away_points)

    home_goals_scored.append(home_gf)
    away_goals_scored.append(away_gf)

    if row['home_score'] > row['away_score']:
        home_points_current = 3
        away_points_current = 0
        
    elif row['home_score'] < row['away_score']:
        home_points_current = 0
        away_points_current = 3
        
    else:
        home_points_current = 1
        away_points_current = 1
    
    
    team_history[home_team].append(
        {
            "points": home_points_current,
            "goals_scored": row['home_score']
        }
    )
    
    team_history[away_team].append(
        {
            "points": away_points_current,
            "goals_scored": row['away_score']
        }
    )
    
    

In [37]:
df['home_form_points_last5'] = home_form_points
df['away_form_points_last5'] = away_form_points

df['home_goals_scored_last5'] = home_goals_scored
df['away_goals_scored_last5'] = away_goals_scored

In [38]:
df['neutral'] = df['neutral'].astype(int)

In [39]:
df[
    [
        "home_team",
        "away_team",
        "home_form_points_last5",
        "away_form_points_last5",
        "home_goals_scored_last5",
        "away_goals_scored_last5",
        "neutral",
        "result"
    ]
].head(20)

,home_team,away_team,home_form_points_last5,away_form_points_last5,home_goals_scored_last5,away_goals_scored_last5,neutral,result
0,Scotland,England,0,0,0.0,0.0,0,1
1,England,Scotland,1,1,0.0,0.0,0,0
2,Scotland,England,1,4,2.0,4.0,0,0
3,England,Scotland,4,4,5.0,4.0,0,1
4,Scotland,England,5,5,6.0,7.0,0,0
5,Scotland,Wales,8,0,9.0,0.0,0,0
6,England,Scotland,5,10,7.0,13.0,0,2
7,Wales,Scotland,0,13,0.0,14.0,0,2
8,Scotland,England,13,4,14.0,8.0,0,0
9,Scotland,Wales,15,0,19.0,0.0,0,0


In [40]:
df.to_csv(
    "../data/features_v1.csv",
    index=False
)

In [41]:
features_df = pd.read_csv(
    "../data/features_v1.csv"
)

features_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_form_points_last5,away_form_points_last5,home_goals_scored_last5,away_goals_scored_last5
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0,1,0,0,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0,0,1,1,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0,0,1,4,2.0,4.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0,1,4,4,5.0,4.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0,0,5,5,6.0,7.0
